<a href="https://colab.research.google.com/github/omrani-wiem/radiographie_classification/blob/main/radiographie_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, re, glob, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, balanced_accuracy_score

SEED = 42
tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = 224
BATCH_SIZE = 32
CLASSES = ["BACTERIAL", "VIRAL"]          # 0 = bactérienne, 1 = virale ; la sortie du modèle est P(virale)
print("GPU :", tf.config.list_physical_devices("GPU"))

GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"omraniwiem","key":"c222f92e3c4d459f57173d7fe7aa6c65"}'}

In [ ]:
!mkdir -p ~/.kaggle && cp kaggle*.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q -o chest-xray-pneumonia.zip -d /content/dataset

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [02:51<00:00, 14.3MB/s]



In [ ]:
ROOT = "/content/dataset/chest_xray"
rows = []
for path in glob.glob(f"{ROOT}/*/PNEUMONIA/*"):
    if not path.lower().endswith((".jpeg", ".jpg", ".png")):
        continue
    name = os.path.basename(path).lower()
    if "bacteria" in name:
        label = "BACTERIAL"
    elif "virus" in name:
        label = "VIRAL"
    else:
        continue                       # type inconnu : on n'invente pas d'étiquette
    m = re.match(r"person(\d+)_", name)
    patient = "P" + (m.group(1) if m else name)
    rows.append((path, label, patient))

df = pd.DataFrame(rows, columns=["path", "label", "patient"])
df["y"] = (df["label"] == "VIRAL").astype("int32")
print(len(df), "images,", df["patient"].nunique(), "patients")
print(df["label"].value_counts())
print("Référence naïve (toujours 'BACTERIAL') :", round((df['label'] == 'BACTERIAL').mean(), 3))


4273 images, 1674 patients
label
BACTERIAL    2780
VIRAL        1493
Name: count, dtype: int64
Référence naïve (toujours 'BACTERIAL') : 0.651


In [ ]:
# Découpage 70 / 15 / 15 par patient, stratifié par classe
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
fold_of = np.zeros(len(df), dtype=int)
for k, (_, idx) in enumerate(sgkf.split(df, df["y"], groups=df["patient"])):
    fold_of[idx] = k
df["split"] = np.where(fold_of < 14, "train", np.where(fold_of < 17, "val", "test"))

assert not (set(df[df.split=="train"].patient) & set(df[df.split=="test"].patient)), "fuite patient train/test"
assert not (set(df[df.split=="train"].patient) & set(df[df.split=="val"].patient)), "fuite patient train/val"
print(pd.crosstab(df["split"], df["label"]))

label  BACTERIAL  VIRAL
split                  
test         412    224
train       1998   1058
val          370    211


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.08),
])

def load(path, y):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32) / 255.0, y          # [0, 1], comme dans le service

def make_ds(part, training):
    d = df[df.split == part]
    ds = tf.data.Dataset.from_tensor_slices((d["path"].values, d["y"].values))
    if training:
        ds = ds.shuffle(len(d), seed=SEED)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds("train", True), make_ds("val", False), make_ds("test", False)

y_train = df[df.split == "train"]["y"].values
w = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weights = {0: float(w[0]), 1: float(w[1])}
print("poids de classes :", class_weights)

poids de classes : {0: 0.7647647647647647, 1: 1.444234404536862}


In [ ]:
def build_model(freeze_backbone=True, dropout=0.3):
    base = keras.applications.EfficientNetB0(include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3), weights="imagenet")
    base.trainable = not freeze_backbone
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = keras.applications.efficientnet.preprocess_input(inp * 255.0)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid", name="p_viral")(x)
    return keras.Model(inp, out), base

model, base = build_model()

def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc")])

cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
      keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)]

compile_model(1e-3)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weights, callbacks=cb)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 90s 622ms/step - accuracy: 0.6243 - auc: 0.6694 - loss: 0.6504 - val_accuracy: 0.5972 - val_auc: 0.7674 - val_loss: 0.6763 - learning_rate: 0.0010
Epoch 2/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 34s 348ms/step - accuracy: 0.7045 - auc: 0.7529 - loss: 0.5920 - val_accuracy: 0.5680 - val_auc: 0.7718 - val_loss: 0.7195 - learning_rate: 0.0010
Epoch 3/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 41s 351ms/step - accuracy: 0.7228 - auc: 0.7865 - loss: 0.5600 - val_accuracy: 0.5938 - val_auc: 0.7716 - val_loss: 0.6832 - learning_rate: 0.0010
Epoch 4/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 40s 413ms/step - accuracy: 0.7310 - auc: 0.7873 - loss: 0.5583 - val_accuracy: 0.6059 - val_auc: 0.7716 - val_loss: 0.6806 - learning_rate: 5.0000e-04
Epoch 5/12
96/96 ━━━━━━━━━━━━━━━━━━━━ 40s 412ms/step - accuracy: 0.7241 - auc: 0.7859 - loss: 0.5593 - val_accuracy: 0.5818 - val_auc: 0.7752 - val_loss: 0.6966 - learning_rate: 5.0000e-04


In [ ]:
# Fine-tuning : on dégèle le dernier tiers du backbone, en gardant les BatchNorm figées
base.trainable = True
cut = int(len(base.layers) * 0.7)
for i, l in enumerate(base.layers):
    l.trainable = i >= cut and not isinstance(l, layers.BatchNormalization)
compile_model(1e-5)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weights, callbacks=cb)

Epoch 1/8
96/96 ━━━━━━━━━━━━━━━━━━━━ 95s 673ms/step - accuracy: 0.7055 - auc: 0.7720 - loss: 0.5755 - val_accuracy: 0.6213 - val_auc: 0.7753 - val_loss: 0.6278 - learning_rate: 1.0000e-05
Epoch 2/8
96/96 ━━━━━━━━━━━━━━━━━━━━ 34s 353ms/step - accuracy: 0.7196 - auc: 0.7793 - loss: 0.5651 - val_accuracy: 0.6609 - val_auc: 0.7777 - val_loss: 0.5935 - learning_rate: 1.0000e-05
Epoch 3/8
96/96 ━━━━━━━━━━━━━━━━━━━━ 35s 358ms/step - accuracy: 0.7320 - auc: 0.7942 - loss: 0.5516 - val_accuracy: 0.6506 - val_auc: 0.7819 - val_loss: 0.5972 - learning_rate: 1.0000e-05
Epoch 4/8
96/96 ━━━━━━━━━━━━━━━━━━━━ 41s 354ms/step - accuracy: 0.7575 - auc: 0.8110 - loss: 0.5316 - val_accuracy: 0.6231 - val_auc: 0.7794 - val_loss: 0.6420 - learning_rate: 1.0000e-05
Epoch 5/8
96/96 ━━━━━━━━━━━━━━━━━━━━ 35s 358ms/step - accuracy: 0.7552 - auc: 0.8141 - loss: 0.5296 - val_accuracy: 0.6059 - val_auc: 0.7883 - val_loss: 0.6745 - learning_rate: 5.0000e-06
Epoch 6/8
96/96 ━━━━━━━━━━━━━━━━━━━━ 34s 349ms/step - accura

In [ ]:
p_viral = model.predict(test_ds, verbose=0).reshape(-1)
y_true = df[df.split == "test"]["y"].values
y_pred = (p_viral >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
print("Matrice de confusion (lignes = vrai, colonnes = prédit) :")
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=CLASSES, columns=CLASSES))
print()
print("Exactitude                   :", round((y_pred == y_true).mean(), 3))
print("Référence naïve (bactérienne):", round((y_true == 0).mean(), 3))
print("Exactitude équilibrée        :", round(balanced_accuracy_score(y_true, y_pred), 3), "(0.5 = hasard)")
print("AUC                          :", round(roc_auc_score(y_true, p_viral), 3), "(0.5 = hasard)")

              precision    recall  f1-score   support

   BACTERIAL      0.836     0.694     0.759       412
       VIRAL      0.571     0.750     0.649       224

    accuracy                          0.714       636
   macro avg      0.704     0.722     0.704       636
weighted avg      0.743     0.714     0.720       636

Matrice de confusion (lignes = vrai, colonnes = prédit) :
           BACTERIAL  VIRAL
BACTERIAL        286    126
VIRAL             56    168

Exactitude                   : 0.714
Référence naïve (bactérienne): 0.648
Exactitude équilibrée        : 0.722 (0.5 = hasard)
AUC                          : 0.804 (0.5 = hasard)


In [ ]:
# Abstention : le service refuse de proposer un type quand sa confiance est trop faible.
# confiance = max(p, 1-p). On regarde, pour chaque seuil, la part des cas où un type est proposé
# et l'exactitude sur ces cas.
conf = np.maximum(p_viral, 1 - p_viral)
print("seuil | type proposé pour | exactitude sur ces cas")
for t in [0.50, 0.55, 0.65, 0.75, 0.85, 0.90]:
    keep = conf >= t
    acc = (y_pred[keep] == y_true[keep]).mean() if keep.any() else float("nan")
    print(f" {t:.2f} |      {keep.mean():5.0%}        |   {acc:.3f}")

seuil | type proposé pour | exactitude sur ces cas
 0.50 |       100%        |   0.714
 0.55 |        88%        |   0.748
 0.65 |        64%        |   0.805
 0.75 |        37%        |   0.850
 0.85 |        10%        |   0.864
 0.90 |         2%        |   0.818


In [ ]:
def table(part):
    d = df[df.split == part]
    p = model.predict(make_ds(part, False), verbose=0).reshape(-1)
    y = d["y"].values
    pred = (p >= 0.5).astype(int)
    conf = np.maximum(p, 1 - p)
    print(f"--- {part} ({len(y)} images) ---")
    print("seuil | type proposé pour | exactitude | VIRAL proposés (précision) | BACTERIAL proposés (précision)")
    for t in [0.50, 0.65, 0.75, 0.85]:
        k = conf >= t
        kv, kb = k & (pred == 1), k & (pred == 0)
        acc = (pred[k] == y[k]).mean() if k.any() else float("nan")
        pv = (y[kv] == 1).mean() if kv.any() else float("nan")
        pb = (y[kb] == 0).mean() if kb.any() else float("nan")
        print(f" {t:.2f} | {k.mean():4.0%} ({k.sum():3d}) | {acc:.3f} | {kv.sum():4d} ({pv:.3f}) | {kb.sum():4d} ({pb:.3f})")

table("val")
table("test")

--- val (581 images) ---
seuil | type proposé pour | exactitude | VIRAL proposés (précision) | BACTERIAL proposés (précision)
 0.50 | 100% (581) | 0.661 |  310 (0.523) |  271 (0.819)
 0.65 |  60% (346) | 0.757 |  180 (0.644) |  166 (0.880)
 0.75 |  35% (203) | 0.862 |  112 (0.777) |   91 (0.967)
 0.85 |   8% ( 49) | 0.898 |   28 (0.821) |   21 (1.000)
--- test (636 images) ---
seuil | type proposé pour | exactitude | VIRAL proposés (précision) | BACTERIAL proposés (précision)
 0.50 | 100% (636) | 0.714 |  294 (0.571) |  342 (0.836)
 0.65 |  64% (405) | 0.805 |  188 (0.702) |  217 (0.894)
 0.75 |  37% (233) | 0.850 |  121 (0.777) |  112 (0.929)
 0.85 |  10% ( 66) | 0.864 |   43 (0.814) |   23 (0.957)


In [ ]:
model.export("sm_type")          # SavedModel
!pip install -q tf2onnx onnxruntime
!python -m tf2onnx.convert --saved-model sm_type --output type.onnx --opset 13

Saved artifact at 'sm_type'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_242')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137158710318224: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  137158710316880: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  137159056170384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137158909991184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137158909990992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137158909991760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137158909991568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137158909989456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137158909991376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137158909990800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1

In [ ]:
# Contrôle : l'ONNX doit donner les mêmes probabilités que Keras.
import onnxruntime as ort
sess = ort.InferenceSession("type.onnx", providers=["CPUExecutionProvider"])
x = next(iter(test_ds))[0][:4].numpy()
out = sess.run(None, {sess.get_inputs()[0].name: x})[0]
print("forme de la sortie ONNX :", out.shape)
assert out.shape == (4, 1), "sortie (n, 1) attendue : P(virale)"
np.testing.assert_allclose(out, model.predict(x, verbose=0), atol=1e-3)
print("OK : ONNX identique à Keras")
files.download("type.onnx")

forme de la sortie ONNX : (4, 1)
OK : ONNX identique à Keras


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>